In [ ]:
from libraries import *
from pdex import parallel_differential_expression


In [ ]:
adata = sc.read_h5ad("./../../Data/ComboScreen.h5ad")


In [ ]:
adata = adata[adata.obs[['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) < 3]

In [ ]:
cols = ['ASCL1', 'KLF14', 'NEUROD1', 'NEUROG1', 'NR3C1', 'NTC', 'SIM1',
        'TET2', 'TWIST1', 'VSX1', 'ZNF385A', 'ZNF547', 'ZNF660', 'ZNF776']

def combine_onehot(row):
    # Select all column names where value == 1
    active = [col for col in cols if row[col] == 1]
    # Join multiple actives with '+', or return 'None' if none are active
    return '+'.join(active) if active else 'None'

adata.obs['perturbation'] = adata.obs[cols].apply(combine_onehot, axis=1)


In [ ]:
sc.pp.normalize_total(adata, target_sum=20000)
sc.pp.log1p(adata)


In [ ]:
adata.obs["perturbation_time"] = (
    adata.obs["perturbation"].astype(str) + "_" + adata.obs["time_point"].astype(str)
)

In [ ]:
adata_day10 = adata[adata.obs["time_point"] == "day10",]

In [ ]:
degs_day10 = parallel_differential_expression(adata_day10, 
                                    groupby_key="perturbation_time", 
                                    reference="NTC_day10", 
                                    is_log1p=True, 
                                    num_workers=128 )
pd.DataFrame(degs_day10).to_csv("Day10DEGs.csv")

In [ ]:
for elem in adata_day10.obs['final_label'].unique():
    adata_day10_tmp = adata_day10[adata_day10.obs['final_label']==elem,:]
    degs_day10_tmp = parallel_differential_expression(adata_day10_tmp, 
                                        groupby_key="perturbation_time", 
                                        reference="NTC_day10", 
                                        is_log1p=True, 
                                        num_workers=128 )
    pd.DataFrame(degs_day10_tmp).to_csv("Day10DEGs_"+elem+".csv")

In [ ]:
adata_day10_differentiated=adata_day10[adata_day10.obs['final_label'].isin(['Differentiated_1', 'Differentiated-2']),:]
degs_day10_tmp = parallel_differential_expression(adata_day10_differentiated, 
                                        groupby_key="perturbation_time", 
                                        reference="NTC_day10", 
                                        is_log1p=True, 
                                        num_workers=128 )
pd.DataFrame(degs_day10_tmp).to_csv("Day10DEGs_DifferentiatedALL.csv")